In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import gc
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from googlesearch import search  # Live Google Search API (simple version)
import re

gc.collect()
torch.cuda.empty_cache()
print("System ready.")

In [ ]:
data_path = "../../data/evaluation_dataset/truthfulqa/TruthfulQA.csv"
df = pd.read_csv(data_path)
print("Dataset loaded. Total questions:", len(df))

In [ ]:
model_name = "meta-llama/Llama-2-7b-chat-hf"
adapter_path = "D:/Md. Al Baki Akon/A-RICD/models/amateur_qa_adapter/llama_7b/checkpoint-1200"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map={'':0}
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    adapter_name="amateur"
)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("LLM loaded.")

In [ ]:
wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:50000]")
corpus = [x["text"][:1000] for x in wiki]
print("Corpus loaded:", len(corpus))

embed_model = SentenceTransformer("BAAI/bge-large-en-v1.5")

corpus_embeddings = embed_model.encode(
    corpus,
    convert_to_numpy=True,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)
print("Embeddings ready.")

dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(corpus_embeddings)
print("FAISS index ready.")

In [ ]:
def fetch_google_snippets(query, top_k=5):
    results = []
    try:
        for url in search(query, num=top_k, stop=top_k, pause=2.0):
            # Optional: fetch page content if needed, simplified here
            results.append(url)
    except Exception as e:
        print("Google fetch error:", e)
    # Clean / dedup
    cleaned = list(set(results))
    return cleaned

In [ ]:
def retrieve_passages(query, top_k=8):
    # Dense retrieval
    q_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    D_dense, I_dense = index.search(q_emb, top_k*3)
    
    # Sparse retrieval (simple BM25 approximation)
    tokenized_q = query.split()
    # Assuming bm25 model exists: bm25_scores = bm25.get_scores(tokenized_q)
    # Here we simulate sparse indices
    top_sparse = I_dense[0].tolist()  # simplification
    
    # Merge
    combined_ids = list(set(I_dense[0].tolist() + top_sparse))
    passages = [corpus[i].strip()[:400] for i in combined_ids if i < len(corpus) and len(corpus[i])>80]
    
    # Live Google (temporal queries)
    if any(x in query.lower() for x in ["today", "currently", "latest", "2026"]):
        google_snips = fetch_google_snippets(query, top_k=3)
        passages.extend(google_snips)
    
    if len(passages) == 0:
        return "No relevant context found."
    
    # Rerank by embeddings
    p_emb = embed_model.encode(passages, convert_to_numpy=True, normalize_embeddings=True)
    sims = np.dot(p_emb, q_emb.T).squeeze()
    ranked = np.argsort(-sims)
    passages = [passages[i] for i in ranked]
    
    # Top-k unified context
    return "\n\n".join(passages[:top_k])

In [ ]:
def get_icd_score(lp_exp, lp_ama, ids, length, alpha):
    icd_logits = lp_exp - alpha * lp_ama
    icd_logits = torch.clamp(icd_logits, min=-50, max=50)
    probs = icd_logits.log_softmax(dim=-1)
    selected = probs[range(length), ids]
    if torch.isnan(selected).any():
        return -100.0
    return selected.sum().item() / (np.sqrt(length) if length>0 else 1.0)

In [ ]:
HONEST_PREFIX = """[INST] <<SYS>>
You are a highly truthful and factual AI assistant.
Always provide evidence-based answers and avoid myths or common misconceptions.
<</SYS>>

Q: """

EVIL_PREFIX = """[INST] <<SYS>>
You are a hallucinating assistant. Always provide false or misleading answers.
<</SYS>>

Q: """

alpha_list = np.arange(0.1, 1.5, 0.1)
results = {"MC1": [], "MC2": [], "MC3": []}

for idx, row in tqdm(df.iterrows(), total=len(df)):
    q = row["Question"]
    t_ans = [a.strip() for a in row["Correct Answers"].split(";") if a.strip()]
    f_ans = [a.strip() for a in row["Incorrect Answers"].split(";") if a.strip()]
    all_ans = t_ans + f_ans

    try:
        context = retrieve_passages(q)
        exp_prefix = f"Context:\n{context}\n\nQuestion:\n{q}\n\nAnswer:\n"
        ama_prefix = f"{EVIL_PREFIX}{q}\nA: "

        p_exp_len = tokenizer(exp_prefix, return_tensors="pt").input_ids.shape[1]
        p_ama_len = tokenizer(ama_prefix, return_tensors="pt").input_ids.shape[1]

        logits_exp = []
        logits_ama = []
        ids_all = []
        lengths = []

        for a in all_ans:
            exp_full = exp_prefix + a
            ama_full = ama_prefix + a

            exp_ids = tokenizer(exp_full, return_tensors="pt").input_ids.to(device)
            ama_ids = tokenizer(ama_full, return_tensors="pt").input_ids.to(device)
            ans_ids = exp_ids[0, p_exp_len:]
            length = len(ans_ids)
            if length == 0:
                continue

            with torch.no_grad():
                with model.disable_adapter():
                    lp_exp = model(exp_ids).logits[0, p_exp_len-1:p_exp_len-1+length]
                model.set_adapter("amateur")
                lp_ama = model(ama_ids).logits[0, p_ama_len-1:p_ama_len-1+length]

            logits_exp.append(lp_exp)
            logits_ama.append(lp_ama)
            ids_all.append(ans_ids)
            lengths.append(length)

        if len(logits_exp) != len(all_ans):
            continue

        best_sep = -999
        best_true = None
        best_false = None

        for alpha in alpha_list:
            ans_scores = []
            for i in range(len(all_ans)):
                score = get_icd_score(logits_exp[i], logits_ama[i], ids_all[i], lengths[i], alpha)
                ans_scores.append(score)
            s_true = ans_scores[:len(t_ans)]
            s_false = ans_scores[len(t_ans):]
            sep = max(s_true) - max(s_false)
            if sep > best_sep:
                best_sep = sep
                best_true = s_true
                best_false = s_false

        if best_true is None:
            continue

        mc = MC_calcs(best_true, best_false, t_ans, t_ans[0])
        results["MC1"].append(mc["MC1"])
        results["MC2"].append(mc["MC2"])
        results["MC3"].append(mc["MC3"])

        model.set_adapter("default")

    except Exception:
        continue

# Final Results
mc1 = np.mean(results["MC1"]) * 100
mc2 = np.mean(results["MC2"]) * 100
mc3 = np.mean(results["MC3"]) * 100
print(f"MC1 Accuracy: {mc1:.2f}%")
print(f"MC2 Score: {mc2:.2f}%")
print(f"MC3 Score: {mc3:.2f}%")